In [1]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.3/118.3 kB 3.1 MB/s eta 0:00:00


In [2]:
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.5 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import re
import string
import contractions
from tqdm import tqdm
tqdm.pandas(desc="Progress Bar")

import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

job_descriptions_data = load_dataset('jacob-hugging-face/job-descriptions', split="train")
job_descriptions_df = pd.DataFrame(job_descriptions_data)

df = pd.read_csv('/content/drive/MyDrive/Final year project/pdf_extracted_skills_education.csv')

cv_df = df[~(df['Skills'].isna() & df['Education'].isna())].reset_index(drop=True)
cv_df = cv_df.fillna(value='')
cv_df['CV'] = cv_df['Skills'] + ' ' + cv_df['Education']

def text_cleaning(text: str) -> str:
    if pd.isnull(text):
        return ""
    text = text.lower().strip()
    text = contractions.fix(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\b\d{1,3}[-./]?\d{1,3}[-./]?\d{1,4}\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

job_descriptions = job_descriptions_df['job_description'].apply(text_cleaning)[:15].to_list()
cv_df['CV'] = cv_df['CV'].progress_apply(text_cleaning)
resumes = cv_df['CV'].to_list()




Progress Bar: 100%|██████████| 2469/2469 [00:00<00:00, 5390.08it/s]


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
cv_df = df[~(df['Skills'].isna() & df['Education'].isna())].reset_index(drop=True)

cv_df = cv_df.fillna(value='')

cv_df['CV'] = cv_df['Skills'] + ' ' + cv_df['Education']

cv_df['CV'] = cv_df['CV'].progress_apply(text_cleaning)

Progress Bar: 100%|██████████| 2469/2469 [00:00<00:00, 3785.77it/s]


In [18]:
%%time

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')


job_description_embeddings = []
for description in job_descriptions:
    tokens = tokenizer(description, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        output = model(**tokens)
    embeddings = output.last_hidden_state.mean(dim=1).numpy()
    job_description_embeddings.append(embeddings[0])

resume_embeddings = []
for resume in resumes:
    tokens = tokenizer(resume, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        output = model(**tokens)
    embeddings = output.last_hidden_state.mean(dim=1).numpy()
    resume_embeddings.append(embeddings[0])

CPU times: user 6min 31s, sys: 507 ms, total: 6min 32s
Wall time: 6min 43s


In [19]:
similarity_scores = cosine_similarity(job_description_embeddings, resume_embeddings)
similarity_scores

array([[0.8157605 , 0.77677274, 0.78489166, ..., 0.8589793 , 0.7705458 ,
        0.60554785],
       [0.7823306 , 0.7171359 , 0.7481306 , ..., 0.8324866 , 0.78843874,
        0.67844903],
       [0.8174368 , 0.7895926 , 0.78363687, ..., 0.84983325, 0.7580298 ,
        0.6202629 ],
       ...,
       [0.815508  , 0.7708844 , 0.7748318 , ..., 0.8843261 , 0.7708576 ,
        0.6345933 ],
       [0.84151214, 0.77827585, 0.79714584, ..., 0.87831897, 0.8251981 ,
        0.68138146],
       [0.83217585, 0.7873483 , 0.7752359 , ..., 0.88907975, 0.7842254 ,
        0.6448566 ]], dtype=float32)

In [20]:
num_top_candidates = 5
top_candidates = []

for i, job_description in enumerate(job_descriptions):
    candidates_with_scores = list(enumerate(similarity_scores[i]))
    candidates_with_scores.sort(key=lambda x: x[1], reverse=True)
    top_candidates_for_job = candidates_with_scores[:num_top_candidates]
    top_candidates.append(top_candidates_for_job)

for i, job_description in enumerate(job_descriptions):
    print(f"Top candidates for JD {i+1} - Postition: {job_descriptions_df['position_title'][i]}")
    for candidate_index, score in top_candidates[i]:
        print(f"  Candidate {candidate_index + 1} - Similarity Score: {score:.4f} - {cv_df['Category'][candidate_index]}/{cv_df['ID'][candidate_index]}.pdf")
    print()

Top candidates for JD 1 - Postition: Sales Specialist
  Candidate 1949 - Similarity Score: 0.9423 - HR/18827609.pdf
  Candidate 291 - Similarity Score: 0.9388 - AGRICULTURE/62994611.pdf
  Candidate 28 - Similarity Score: 0.9377 - ACCOUNTANT/16237710.pdf
  Candidate 478 - Similarity Score: 0.9376 - ARTS/43622023.pdf
  Candidate 1803 - Similarity Score: 0.9320 - HEALTHCARE/10466208.pdf

Top candidates for JD 2 - Postition: Apple Solutions Consultant
  Candidate 168 - Similarity Score: 0.9236 - ADVOCATE/22391901.pdf
  Candidate 904 - Similarity Score: 0.9165 - BUSINESS-DEVELOPMENT/95382114.pdf
  Candidate 1730 - Similarity Score: 0.9159 - FITNESS/21238396.pdf
  Candidate 952 - Similarity Score: 0.9155 - CHEF/21869994.pdf
  Candidate 482 - Similarity Score: 0.9146 - ARTS/54100393.pdf

Top candidates for JD 3 - Postition: Licensing Coordinator - Consumer Products
  Candidate 478 - Similarity Score: 0.9589 - ARTS/43622023.pdf
  Candidate 2153 - Similarity Score: 0.9499 - PUBLIC-RELATIONS/122

In [23]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


thresholds = np.arange(0.1, 0.91, 0.01)


ground_truth = np.random.randint(0, 2, similarity_scores.shape)


best_threshold = 0
best_f1 = 0
results = []

for threshold in thresholds:
    predictions = (similarity_scores >= threshold).astype(int)

    y_true = ground_truth.flatten()
    y_pred = predictions.flatten()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    results.append((threshold, acc, prec, rec, f1))

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

In [22]:
ground_truth = np.zeros_like(similarity_scores)

for i in range(len(job_descriptions)):
    ground_truth[i][0] = 1
threshold = 0.89
predictions = (similarity_scores >= threshold).astype(int)
correct_predictions = (predictions == ground_truth).sum()
total_predictions = ground_truth.size
accuracy = (correct_predictions / total_predictions)*100
print(f"Accuracy: {int(accuracy)}%")

Accuracy: 94%
